In [1]:
from pathlib import Path
from datetime import datetime
from typing import List
from types import SimpleNamespace
import pickle
import numpy as np
import torch
from pymatgen.core import Structure, Lattice, Species, Element


from chggen.pl_data.dataset import CHGNetDataset
from chggen.pl_modules.model import CHGGen
from chggen.common.data_utils import get_scaler_from_data_list
from chggen.pl_modules.decoder import NequipDecoder

from torch_geometric.data import Data
from torch import nn
import torch.nn.functional as F



def get_scaler(dataset, use_prop_scaler = False, 
               scaler_path = None):
    # Load once to compute property scaler
    if scaler_path is None:
        lattice_scaler = get_scaler_from_data_list(
            dataset.cached_data,
            key='scaled_lattice')
        if use_prop_scaler:
            NotImplementedError("Not implemented the multi prop scaler yet.")
    else:
        lattice_scaler = torch.load(
            Path(scaler_path) / 'lattice_scaler.pt')
    return lattice_scaler



/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
chggen = CHGGen.load_from_checkpoint(checkpoint_path="./test_models/perov/trainer_perov.ckpt")



CHGNet initialized with 400,438 parameters
CHGNet initialized with 400,438 parameters


/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/torch/jit/_check.py:172: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn("The TorchScript type system doesn't support "


In [3]:
with open('./test_models/lattice_scaler_perov', 'rb') as fp:
    lattice_scaler = pickle.load(fp)
    
chggen.lattice_scaler = lattice_scaler

In [4]:
# with open('./test_models/chggen_perov.pk', 'rb') as fp:
#     chggen = pickle.load(fp)

In [5]:
# for paras in chggen.encoder.parameters():
#     print(paras)
#     print(paras.requires_grad)

In [37]:
chggen.sigmas

Parameter containing:
tensor([1.0000e+01, 8.6851e+00, 7.5431e+00, 6.5513e+00, 5.6899e+00, 4.9417e+00,
        4.2919e+00, 3.7276e+00, 3.2375e+00, 2.8118e+00, 2.4421e+00, 2.1210e+00,
        1.8421e+00, 1.5999e+00, 1.3895e+00, 1.2068e+00, 1.0481e+00, 9.1030e-01,
        7.9060e-01, 6.8665e-01, 5.9636e-01, 5.1795e-01, 4.4984e-01, 3.9069e-01,
        3.3932e-01, 2.9471e-01, 2.5595e-01, 2.2230e-01, 1.9307e-01, 1.6768e-01,
        1.4563e-01, 1.2649e-01, 1.0985e-01, 9.5410e-02, 8.2864e-02, 7.1969e-02,
        6.2506e-02, 5.4287e-02, 4.7149e-02, 4.0949e-02, 3.5565e-02, 3.0888e-02,
        2.6827e-02, 2.3300e-02, 2.0236e-02, 1.7575e-02, 1.5264e-02, 1.3257e-02,
        1.1514e-02, 1.0000e-02])

In [56]:
# langevin dynamics
ld_kwargs = SimpleNamespace(n_step_each = 10,
                            step_lr = 1e-4,
                            min_sigma = 0,
                            save_traj = True,
                            disable_bar = False,
                            compute_force = False,
                            beta_c = 0, 
                            beta_f = 1e-3)

z = torch.rand(1, 64, requires_grad= True,)
results = chggen.langevin_dynamics_guidance(z = z, 
                                            prop_guidance = torch.tensor(-0.05), 
                                            gt_num_atoms= torch.tensor([12]), 
                                            gt_atom_types= torch.tensor([3,3,3, 25,25,25, 8,8,8,8,8,8]), 
                                            ld_kwargs= ld_kwargs)


/home/zhongpc/chggen/chggen/common/data_utils.py:625: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)
  0%|                                                                                                                          | 0/50 [00:00<?, ?it/s]

LiMnO2
LiMnO2
LiMnO2
LiMnO2
LiMnO2
LiMnO2
LiMnO2
LiMnO2


  2%|██▎                                                                                                               | 1/50 [00:01<01:06,  1.36s/it]

LiMnO2
LiMnO2
LiMnO2
LiMnO2
LiMnO2
LiMnO2
LiMnO2
LiMnO2
LiMnO2
LiMnO2
LiMnO2


  4%|████▌                                                                                                             | 2/50 [00:02<00:53,  1.12s/it]

...

 96%|████████████████████████████████████████████████████████████████████████████████████████████████████████████▍    | 48/50 [00:43<00:01,  1.17it/s]

LiMnO2
LiMnO2
LiMnO2
LiMnO2
LiMnO2
LiMnO2
LiMnO2
LiMnO2
LiMnO2


 98%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████▋  | 49/50 [00:44<00:00,  1.19it/s]

LiMnO2
LiMnO2
LiMnO2
LiMnO2
LiMnO2
LiMnO2
LiMnO2
LiMnO2
LiMnO2
LiMnO2
LiMnO2
LiMnO2


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:45<00:00,  1.10it/s]


In [57]:
# save the results from langevin dynamics
lengths = results['lengths']
angles= results['angles']
num_atoms = results['num_atoms']
frac_coords = results['frac_coords']
atom_types = results['atom_types']

batch = torch.arange(len(num_atoms))
batch = batch.repeat_interleave(num_atoms)

for ii in range(len(num_atoms)):
    indices = torch.where(batch == ii)[0]
    print(ii, indices)
    if len(indices) == 0:
        continue
    
    
    Latt = Lattice.from_parameters(a = lengths[ii,0], b = lengths[ii,1], c = lengths[ii,2],
                                   alpha= angles[ii, 0], beta= angles[ii,1], gamma=angles[ii, 2])
                                   
    frac_ = frac_coords[indices]
    type_ = atom_types[indices]
    species_ = [Element.from_Z(ele_Z) for ele_Z in type_]
    
    s_gen = Structure(lattice= Latt , species= species_, coords= frac_.detach().numpy(),
                      to_unit_cell=False,coords_are_cartesian=False);
    print(s_gen.composition)
    s_gen.to(filename= './test_models/structures/prop_guidance_' + str(ii) + '.cif')
    

0 tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
Li3 Mn3 O6


In [58]:
# save the results from langevin dynamics
lengths = results['lengths']
angles= results['angles']
num_atoms = results['num_atoms']
all_frac_coords = results['all_frac_coords']
all_atom_types = results['all_atom_types']

batch = torch.arange(len(num_atoms))
batch = batch.repeat_interleave(num_atoms)

for ii in range(len(num_atoms)):
    indices = torch.where(batch == ii)[0]
    print(ii, indices)
    if len(indices) == 0:
        continue
    
    
    Latt = Lattice.from_parameters(a = lengths[ii,0], b = lengths[ii,1], c = lengths[ii,2],
                                   alpha= angles[ii, 0], beta= angles[ii,1], gamma=angles[ii, 2])
    
    for jj in range(0, len(all_frac_coords), 1):
        
                                   
        frac_ = all_frac_coords[jj, indices, :]
        type_ = all_atom_types[jj, indices]
        species_ = [Element.from_Z(ele_Z) for ele_Z in type_]

        s_gen = Structure(lattice= Latt , species= species_, coords= frac_.detach().numpy(),
                          to_unit_cell=False,coords_are_cartesian=False);
        print(s_gen.composition)
        s_gen.to(filename= './test_models/structures/traj_guidance_' + str(ii) + '_traj_' + str(jj) + '.cif')


0 tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn3 O6
Li3 Mn

In [59]:
all_frac_coords.shape

torch.Size([500, 12, 3])

In [60]:
indices

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])

In [26]:
all_frac_coords[jj, indices, :]

tensor([[0.2896, 0.3446, 0.0266],
        [0.5777, 0.5927, 0.7847],
        [0.3187, 0.3374, 0.0976],
        [0.2204, 0.1796, 0.2935],
        [0.5935, 0.5595, 0.9657]], grad_fn=<IndexBackward0>)

In [27]:
all_atom_types.shape

torch.Size([500, 5])